# Étude de clustering des destinations — TravelMatch

**Objectif** : segmenter les destinations en *archétypes d'activités* interprétables, qui servent de "type de destination" dans le moteur de recommandation.

**Jeu de données** : `DATA/raw/Worldwide_Travel_Cities.csv` — **560 villes**, **0 valeur manquante**, **9 dimensions d'activités** notées de 1 à 5 (culture, adventure, nature, beaches, nightlife, cuisine, wellness, urban, seclusion).

**Choix de conception (cohérence)** :
- On **clusterise uniquement sur les 9 activités** (l'ADN de la destination).
- Le budget, le climat et la région servent **seulement à profiler/expliquer** les clusters, pas à les former. Raison : le recommandeur note déjà séparément la température et le budget — les inclure ici reviendrait à compter deux fois la même information.

Toute la logique vit dans `mlops/` (versionnée, testée). Ce notebook ne fait que l'orchestrer et commenter les résultats.

In [ ]:
import os, sys
ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)
os.chdir(ROOT)

from mlops.features import build_destination_features, ACTIVITY_FEATURES
from mlops.train import run_clustering

## 1. Les données

In [ ]:
cities, X, descriptive = build_destination_features()
print(f"{len(cities)} villes · {len(ACTIVITY_FEATURES)} dimensions d'activités")
X.describe().loc[["min", "max", "mean", "std"]].round(2)

## 2. Méthodologie

1. **Standardisation** des 9 activités (`StandardScaler`) — elles ont des variances différentes.
2. **KMeans** pour `k` de 2 à 12.
3. **Sélection de k** par 4 indicateurs : méthode du coude (inertie), silhouette, Davies-Bouldin, Calinski-Harabasz.
4. **Arbitrage** : les métriques internes informent, le métier décide (cf. ci-dessous).
5. **Profilage** de chaque cluster pour le rendre explicable.

L'appel ci-dessous logge tout dans MLflow (experiment `TravelMatch_Clustering`) : courbes de sélection, diagramme de silhouette, heatmap de l'ADN, projection PCA, tailles, et table des profils.

In [ ]:
# final_k=6 : on retient 6 archétypes actionnables (voir l'arbitrage en section 4)
chosen_k = run_clustering(k_min=2, k_max=12, final_k=6)
print("k retenu :", chosen_k)

## 3. Lecture des résultats

Les profils (label auto = top-2 activités distinctives, villes représentatives, budget/climat/région) sont imprimés ci-dessus et sauvegardés dans MLflow (`reports/cluster_profiles.csv`).

Les 6 archétypes obtenus :

| Cluster | Archétype | Lecture métier | Villes types |
|---|---|---|---|
| C0 | urban + nightlife | Métropoles animées | Baku, Toulouse, Bristol |
| C1 | adventure + nature | Nature & aventure | Yangshuo, Bled, Guilin |
| C2 | beaches + wellness | Plages & bien-être (tropical) | Port Vila, Nadi |
| C3 | culture + cuisine | Culture & gastronomie (abordable) | Granada, Cork |
| C4 | beaches + nightlife | Plages festives | Tampa, Gold Coast |
| C5 | seclusion + culture | Hors des sentiers battus | Skopje, Harare |

## 4. Arbitrage sur le nombre de clusters (point clé pour le jury)

La **silhouette est maximale à k=2** (≈ 0.29) et décroît ensuite : sur des notes d'activités, les destinations forment un **continuum**, pas des groupes nettement séparés. La silhouette à k=2 oppose simplement *« nature / wellness / calme »* vs *« culture / vie urbaine »*.

Mais **k=2 n'est pas actionnable** pour un produit de recommandation. On retient donc **k=6** sur un critère **métier** : 6 archétypes équilibrés (66 à 143 villes), interprétables et utiles à l'utilisateur. C'est un choix **assumé et documenté** — les courbes de sélection (dans MLflow) justifient la discussion.

> *Règle de l'art : les indices internes orientent, mais le nombre de clusters d'un produit se décide aussi sur l'interprétabilité et l'usage.*

**Limites / pistes** : structure de clusters faible (continuum) → on pourrait (a) enrichir l'ADN avec d'autres signaux d'activité, (b) tester une segmentation hiérarchique, (c) mesurer la stabilité par bootstrap (ARI). Pour visualiser les artefacts : ouvrir l'UI MLflow sur http://localhost:5000.